# FLUX.2 Klein — standalone Gradio notebook

**Kernel:** `phase2_3d`; project root `~/OEC`. Run top to bottom. This notebook delegates model loading, PNG/JSON output validation and storage admission to `pipeline_flux_image.py`, the same reusable CLI that can run in `check`, `generate` or `gradio` mode.

The loader uses the cached `black-forest-labs/FLUX.2-klein-4B` snapshot first. A missing model or dependency cannot be downloaded/installed unless OEC StorageManager approves a fresh quota snapshot. Output writes under `SYSTEM_DIRS['outputs']/images` also use guarded storage admission. Public Gradio sharing remains disabled.

The UI defaults to port **7864**; tunnel it with `ssh -N -L 7864:COMPUTE_NODE:7864 rahkus@login.npsf.cdac.in` after replacing `COMPUTE_NODE` with the hostname printed below.

In [ ]:
# Load OEC paths first; core_config does not import PyTorch.
import core_config
from core_config import SYSTEM_DIRS
import os

target_gpu = os.environ.get('AIEROS_GPU')
if target_gpu is None:
    target_gpu = os.environ.get('CUDA_VISIBLE_DEVICES', '0').split(',')[0].strip()
if not target_gpu or not target_gpu.isdigit():
    raise ValueError(
        f'AIEROS_GPU/CUDA_VISIBLE_DEVICES must identify a GPU index, got {target_gpu!r}'
    )
os.environ['CUDA_VISIBLE_DEVICES'] = target_gpu
print(f'[SYSTEM] Selected GPU setting: {target_gpu}; this kernel uses cuda:0.')
print(f'[SYSTEM] Model cache: {SYSTEM_DIRS["models"]}')
print(f'[SYSTEM] Output root: {SYSTEM_DIRS["outputs"]}')

In [ ]:
# Read-only process inspection; do not kill a fixed PID on a shared node.
!ps -u "$USER" -o pid,lstart,etime,comm | grep -E '(^ *PID|python|jupyter)' || true

In [ ]:
import socket
print('COMPUTE_NODE=' + socket.gethostname())
!nvidia-smi

In [ ]:
# Read-only Jupyter/kernel dependency check. Do not upgrade the shared notebook stack here.
import importlib.metadata as metadata
import sys

print('ACTIVE_KERNEL_PYTHON=' + sys.executable)
for package in ('jupyter', 'ipykernel', 'ipywidgets', 'gradio'):
    try:
        print(f'{package}={metadata.version(package)}')
    except metadata.PackageNotFoundError:
        print(f'{package}=MISSING')

In [ ]:
import importlib.metadata as metadata
import os
import subprocess
import sys

REQUIRED_FLUX_PACKAGES = {
    'diffusers': '0.37.0',
    'huggingface-hub': '0.36.0',
    'transformers': '4.56.1',
}
missing_or_mismatched = {}
for package, required_version in REQUIRED_FLUX_PACKAGES.items():
    try:
        installed_version = metadata.version(package)
    except metadata.PackageNotFoundError:
        installed_version = None
    if installed_version != required_version:
        missing_or_mismatched[package] = required_version
    print(f'FLUX_PACKAGE_{package}={installed_version or "MISSING"} required={required_version}')

if missing_or_mismatched:
    # The install writes into a shared conda environment, so it requires fresh
    # filesystem headroom and uses OEC's reserve-preserving managed-operation lock.
    from storage_manager import managed_storage_operation, storage_preflight

    setup_budget_gb = max(
        1.0, float(os.environ.get('AIEROS_FLUX_SETUP_BUDGET_GB', '2.0'))
    )
    ok, reason = storage_preflight(
        'models',
        required_gb=setup_budget_gb,
        require_fresh_quota=True,
        minimum_free_gb=2.0,
    )
    print(f'FLUX_SETUP_STORAGE_ADMISSION={ok} reason={reason}')
    if not ok:
        raise RuntimeError(
            f'FLUX dependency install denied by OEC storage policy: {reason}. '
            'Obtain a fresh authoritative quota snapshot and satisfy StorageManager admission before rerunning this cell.'
        )
    ok, reason = storage_preflight(
        "models",
        required_gb=setup_budget_gb,
        require_fresh_quota=True,
        minimum_free_gb=2.0,
    )
    if not ok:
        raise RuntimeError(f"Storage admission denied before managed write: {reason}")
    with managed_storage_operation(
        'models',
        required_gb=setup_budget_gb,
        require_fresh_quota=True,
        minimum_free_gb=2.0,
    ):
        ok, reason = storage_preflight(
            "models",
            required_gb=setup_budget_gb,
            require_fresh_quota=True,
            minimum_free_gb=2.0,
        )
        if not ok:
            raise RuntimeError(f"Storage admission denied inside managed write: {reason}")
        subprocess.run(
            [
                sys.executable, '-m', 'pip', 'install',
                'diffusers==0.37.0',
                'huggingface-hub==0.36.0',
                'transformers==4.56.1',
            ],
            check=True,
        )
else:
    print('FLUX_DEPENDENCY_INSTALL=SKIPPED versions_already_match=True')

# Verify exact installed releases and eagerly import the Klein pipeline.
subprocess.run(
    [
        sys.executable, '-c',
        (
            "import importlib.metadata as m; "
            "assert m.version('diffusers') == '0.37.0', m.version('diffusers'); "
            "assert m.version('huggingface-hub') == '0.36.0', m.version('huggingface-hub'); "
            "assert m.version('transformers') == '4.56.1', m.version('transformers'); "
            "from diffusers import DiffusionPipeline, Flux2KleinPipeline; "
            "print('FLUX_IMPORT_CHECK=PASS', "
            "'diffusers='+m.version('diffusers'), "
            "'huggingface_hub='+m.version('huggingface-hub'))"
        ),
    ],
    check=True,
)
print('FLUX.2 dependency stack is ready.')

In [ ]:
# Reuse the standalone FLUX CLI implementation so notebook and script behavior match.
from pipeline_flux_image import load_flux_pipeline, create_gradio_demo

FLUX_SERVER_PORT = int(os.environ.get("AIEROS_FLUX_GRADIO_PORT", "7864"))
if "flux_pipe" not in globals() or flux_pipe is None:
    flux_pipe = load_flux_pipeline(
        model_id="black-forest-labs/FLUX.2-klein-4B",
        model_cache=SYSTEM_DIRS["models"],
        offline=True,
        gpu=target_gpu,
    )
else:
    print("FLUX_PIPELINE=REUSED_EXISTING_INSTANCE")

# Close an older Gradio server if this cell is re-run in the same kernel.
if "demo_flux" in globals():
    try:
        demo_flux.close()
    except Exception:
        pass
demo_flux = create_gradio_demo(
    flux_pipe,
    output_dir=os.path.join(SYSTEM_DIRS["outputs"], "images"),
    port=FLUX_SERVER_PORT,
)
print(f"FLUX_PIPELINE_READY=PASS model=black-forest-labs/FLUX.2-klein-4B; port={FLUX_SERVER_PORT}")
demo_flux.launch(
    server_name="0.0.0.0", server_port=FLUX_SERVER_PORT,
    inline=False, share=False, show_error=True,
)

In [ ]:
# demo.close()

In [ ]:
!nvidia-smi